# Urban ML Pipeline — Analysis

This notebook analyzes the dataset generated by `run_pipeline.py` and runs the full ML workflow required by the DEML course:

1. **EDA** — Class distribution, boxplots, correlations, SOM
2. **Dimensionality reduction** — PCA (biplot), ICA (mixing matrix), t-SNE
3. **Encoding experiments** — Scaling (StandardScaler vs MinMaxScaler vs none) and target encoding
4. **Supervised classification** — Logistic Regression (baseline), XGBoost, Random Forest, SVC, ANN
5. **Ablation study + tuning** — Remove ONE feature at a time, GridSearchCV
6. **Clustering** — K-Means with elbow + silhouette
7. **Geographic heatmaps** — Spatial visualization of predictions per city
8. **Cross-city comparison** — Feature means, accuracy, class balance across cities
9. **Binary vs 3-Class** — Compare with and without Mixed-Use
10. **Transfer Learning** — Train on Ground Truth cities, predict on OSM-Only cities

### Setup — Imports and Configuration

Loads all required libraries. Optional imports (xgboost, tensorflow, minisom, folium) are in `try/except` blocks — if not installed, the notebook keeps running and simply skips those sections.

Also imports pipeline constants from `config.py`: feature list, class mapping, neural network configuration, etc.

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kurtosis
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.metrics import (classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, accuracy_score)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA, FastICA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.pipeline import Pipeline

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("xgboost not installed — skipping XGBoost")

try:
    import tensorflow as tf
    from tensorflow import keras
    HAS_TF = True
except ImportError:
    HAS_TF = False
    print("tensorflow not installed — skipping ANN")

try:
    from minisom import MiniSom
    HAS_SOM = True
except ImportError:
    HAS_SOM = False
    print("minisom not installed — run: pip install minisom")

try:
    import folium
    HAS_FOLIUM = True
except ImportError:
    HAS_FOLIUM = False
    print("folium not installed — skipping interactive maps")

# Import pipeline config
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))
from config import FEATURE_COLS, get_class_map, CITY_REGISTRY, ANN_CONFIG, KMEANS_MAX_K, TSNE_PERPLEXITY

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100
os.makedirs("outputs", exist_ok=True)
print("Setup complete")

## 0. Data Loading

Loads the combined CSV with all cities. This file is generated by `run_pipeline.py` and contains one row per grid cell (150m x 150m) with its features and the `zone_type` column (the Y label).

**What to look for in the output:**
- Total number of rows (each row = one grid cell)
- List of included cities
- Available column names

### Dataset Preparation

Applies class mapping (`Commercial`, `Mixed-Use` → `Residential`, etc.), selects the features, scales with `StandardScaler`, encodes labels, and splits into train/test (80/20).

**What to look for in the output:**
- **Class distribution:** How many cells per class? If there is heavy imbalance (>5:1), models need `class_weight="balanced"`.
- **Missing features:** If a WARNING appears, some feature was not generated by the pipeline.
- **N_FOLDS:** If the smallest class has few samples, CV automatically uses fewer folds.
- **Train/Test split:** Should be approximately 80/20.

In [ ]:
DATA_PATH = "csv/all_cities_combined.csv"
assert os.path.exists(DATA_PATH), f"Run `python run_pipeline.py` first to generate {DATA_PATH}"

df_raw = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df_raw)} rows, {len(df_raw.columns)} columns")
print(f"Cities: {df_raw['city'].unique().tolist()}")
print(f"Columns: {list(df_raw.columns)}")
df_raw.head()

In [ ]:
# Apply class mapping
class_map = get_class_map()
df = df_raw[df_raw["zone_type"].isin(class_map)].copy()
df["label"] = df["zone_type"].map(class_map)
print(f"After class mapping: {len(df)} rows")
print(f"Class distribution:\n{df['label'].value_counts()}")

# Feature matrix
available_features = [f for f in FEATURE_COLS if f in df.columns]
missing = [f for f in FEATURE_COLS if f not in df.columns]
if missing:
    print(f"WARNING: Missing features: {missing}")
print(f"Using {len(available_features)} features: {available_features}")

X = df[available_features].fillna(0).values
y = df["label"].values
cities = df["city"].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

le = LabelEncoder()
y_encoded = le.fit_transform(y)
class_names = le.classes_.tolist()
print(f"Classes: {class_names}")

# Determine safe number of CV folds based on smallest class
min_class_count = pd.Series(y_encoded).value_counts().min()
N_FOLDS = min(5, min_class_count)
if N_FOLDS < 5:
    print(f"WARNING: Smallest class has {min_class_count} samples — using {N_FOLDS}-fold CV instead of 5")

# Stratified split (need at least 2 per class)
if min_class_count >= 2:
    X_train, X_test, y_train, y_test, cities_train, cities_test = train_test_split(
        X_scaled, y_encoded, cities, test_size=0.2, random_state=42, stratify=y_encoded
    )
else:
    print("WARNING: Class too small for stratified split — using random split")
    X_train, X_test, y_train, y_test, cities_train, cities_test = train_test_split(
        X_scaled, y_encoded, cities, test_size=0.2, random_state=42
    )
print(f"Train: {len(X_train)}, Test: {len(X_test)}")

### Boxplots — Distribution of Each Feature by Class

One boxplot per feature, separated by class (Commercial vs Residential).

**How to read a boxplot:**
- The **box** contains the central 50% of the data (Q1 to Q3)
- The **green line** inside the box = median
- The **whiskers** extend to 1.5x the interquartile range
- **Circles** outside the whiskers = outliers

**What to look for:**
- If the Commercial and Residential boxes **do not overlap** → that feature separates the classes well (good feature)
- If they overlap a lot → that feature alone does not distinguish the classes (but may still be useful in combination with others)
- **Outliers** in one class but not the other → the feature captures extreme behavior in that class

### Correlation Matrix

Shows the Pearson coefficient (r) between each pair of features. Values from -1 to 1.

**How to interpret:**
- **r > 0.7** (dark red): Highly correlated features → probably redundant (measure the same thing). Candidates to remove one of the two.
- **r ≈ 0** (white): Independent features → each contributes different information. Ideal.
- **r < -0.7** (dark blue): Strong negative correlation → also redundant (one is the "inverse" of the other).
- **The diagonal** always has 1.0 (each feature is perfectly correlated with itself).

**Rule of thumb:** If no pair exceeds |r| > 0.7, all features are sufficiently independent.

## 1. Exploratory Data Analysis (EDA)

EDA is the mandatory first step before training any model. Its goal is to **understand the data visually** before making decisions.

**Professor's rule:** "Visualize the data BEFORE training — plots, plots, plots."

In this section we generate:
- **Class distribution** — Is the dataset balanced? How many cells per class and per city?
- **Feature boxplots** — Which features separate the classes well? Which ones overlap?
- **Correlation matrix** — Are there redundant features (r > 0.7)? Which are independent?

In [ ]:
# Class distribution per city
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df["label"].value_counts().plot.bar(ax=axes[0], color=["steelblue", "coral", "green"][:len(class_names)])
axes[0].set_title("Overall Class Distribution")
axes[0].set_ylabel("Count")

ct = pd.crosstab(df["city"], df["label"])
ct.plot.bar(ax=axes[1], color=["steelblue", "coral", "green"][:len(class_names)])
axes[1].set_title("Class Distribution per City")
axes[1].set_ylabel("Count")
axes[1].legend(title="Class")

plt.tight_layout()
plt.savefig("outputs/01_class_distribution.png", bbox_inches="tight")
plt.show()

In [ ]:
# Feature boxplots
n_feat = len(available_features)
n_cols_plot = 3
n_rows_plot = (n_feat + n_cols_plot - 1) // n_cols_plot
fig, axes = plt.subplots(n_rows_plot, n_cols_plot, figsize=(15, 4 * n_rows_plot))
axes = axes.flatten()

for i, feat in enumerate(available_features):
    df.boxplot(column=feat, by="label", ax=axes[i])
    axes[i].set_title(feat)
    axes[i].set_xlabel("")

for i in range(n_feat, len(axes)):
    axes[i].set_visible(False)

plt.suptitle("Feature Distributions by Class", y=1.02, fontsize=14)
plt.tight_layout()
plt.savefig("outputs/02_feature_boxplots.png", bbox_inches="tight")
plt.show()

In [ ]:
# Correlation heatmap
corr = df[available_features].corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Feature Correlation Matrix")
plt.tight_layout()
plt.savefig("outputs/03_correlation_heatmap.png", bbox_inches="tight")
plt.show()

### 1.1 Self-Organizing Map (SOM / Kohonen Map)

A SOM is an **unsupervised neural network** that projects high-dimensional data (features) onto a 2D grid. Each neuron in the map "absorbs" the most similar data points. Similar data end up in neighboring neurons.

**What is it for?** To see if the data has natural groupings WITHOUT giving the model the labels. If the SOM groups data similarly to our classes (Commercial vs Residential), that confirms the features are good.

**How to interpret the 3 plots:**
- **U-Matrix (left):** Dark zones = boundaries between groups. Light zones = homogeneous regions. If there is a clear dark "river" dividing the map → there are 2 well-separated groups.
- **SOM with labels (center):** Each neuron colored by the majority class of its assigned points. If Commercial and Residential occupy distinct regions → the features capture the real class differences.
- **Component Planes (right):** A mini-heatmap per feature showing its intensity across the SOM. Features that look similar to the class map are the most discriminating.

In [ ]:
if HAS_SOM:
    # Subsample for SOM (winner mapping is O(n) per sample)
    SOM_MAX = 8000
    if len(X_scaled) > SOM_MAX:
        rng_som = np.random.RandomState(42)
        som_idx = rng_som.choice(len(X_scaled), SOM_MAX, replace=False)
        X_som = X_scaled[som_idx]
        y_som = y[som_idx]
        cities_som = cities[som_idx]
        print(f"SOM: subsampled {SOM_MAX} from {len(X_scaled)}")
    else:
        X_som = X_scaled
        y_som = y
        cities_som = cities
        som_idx = np.arange(len(X_scaled))

    # Train SOM on scaled features
    som_x, som_y = 10, 10  # 10x10 grid
    som = MiniSom(som_x, som_y, X_som.shape[1],
                  sigma=1.5, learning_rate=0.5, random_seed=42)
    som.random_weights_init(X_som)
    som.train_random(X_som, num_iteration=5000)

    # Plot 1: U-Matrix (distance between neighboring neurons)
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))

    umatrix = som.distance_map()
    axes[0].imshow(umatrix.T, cmap="bone_r", origin="lower")
    axes[0].set_title("SOM U-Matrix (dark = cluster boundary)")
    axes[0].set_xlabel("SOM X"); axes[0].set_ylabel("SOM Y")

    # Plot 2: SOM colored by zone type
    zone_colors_map = {"Commercial": "red", "Residential": "blue", "Other": "green"}
    for i, (x_val, label) in enumerate(zip(X_som, y_som)):
        w = som.winner(x_val)
        color = zone_colors_map.get(label, "gray")
        axes[1].plot(w[0] + np.random.uniform(-0.3, 0.3),
                     w[1] + np.random.uniform(-0.3, 0.3),
                     "o", color=color, markersize=2, alpha=0.4)
    axes[1].set_xlim(-0.5, som_x - 0.5)
    axes[1].set_ylim(-0.5, som_y - 0.5)
    axes[1].set_title("SOM — colored by Zone Type")
    for label, color in zone_colors_map.items():
        if label in class_names:
            axes[1].plot([], [], "o", color=color, label=label, markersize=8)
    axes[1].legend()

    # Plot 3: SOM colored by city
    city_colors = plt.cm.tab10(np.linspace(0, 1, len(df["city"].unique())))
    city_list_som = sorted(df["city"].unique())
    city_color_map = {c: city_colors[i] for i, c in enumerate(city_list_som)}
    for i, (x_val, city_name) in enumerate(zip(X_som, cities_som)):
        w = som.winner(x_val)
        axes[2].plot(w[0] + np.random.uniform(-0.3, 0.3),
                     w[1] + np.random.uniform(-0.3, 0.3),
                     "o", color=city_color_map[city_name], markersize=2, alpha=0.4)
    axes[2].set_xlim(-0.5, som_x - 0.5)
    axes[2].set_ylim(-0.5, som_y - 0.5)
    axes[2].set_title("SOM — colored by City")
    for c in city_list_som:
        axes[2].plot([], [], "o", color=city_color_map[c], label=c, markersize=8)
    axes[2].legend(fontsize=8)

    plt.tight_layout()
    plt.savefig("outputs/03b_som.png", bbox_inches="tight")
    plt.show()

    # Component planes
    X_raw_som = X[som_idx] if len(X) > SOM_MAX else X
    n_feat = len(available_features)
    fig, axes = plt.subplots(2, (n_feat + 1) // 2, figsize=(4 * ((n_feat + 1) // 2), 8))
    axes = axes.flatten()
    for i, feat in enumerate(available_features):
        plane = np.zeros((som_x, som_y))
        count = np.zeros((som_x, som_y))
        for x_val, feat_val in zip(X_som, X_raw_som[:, i]):
            w = som.winner(x_val)
            plane[w] += feat_val
            count[w] += 1
        count[count == 0] = 1
        plane /= count
        axes[i].imshow(plane.T, cmap="viridis", origin="lower")
        axes[i].set_title(feat, fontsize=9)
    for i in range(n_feat, len(axes)):
        axes[i].set_visible(False)
    plt.suptitle("SOM Component Planes (feature intensity per neuron)", y=1.02)
    plt.tight_layout()
    plt.savefig("outputs/03c_som_components.png", bbox_inches="tight")
    plt.show()
else:
    print("Skipping SOM — install minisom: pip install minisom")

## 2. Feature Analysis — Dimensionality Reduction

Our data lives in multiple dimensions (one per feature). The human brain can only see in 2D or 3D. Dimensionality reduction techniques **compress** those dimensions to 2 for visualization.

Three complementary techniques:
- **PCA**: Finds the directions of maximum variance. It is **deterministic** (always the same result). Generates a **biplot** with arrows showing which features point in which direction.
- **ICA**: Finds statistically **independent** signals hidden in the data. Useful for identifying sources that are mixed together in the features.
- **t-SNE**: Preserves **local neighborhoods**: points that are close in high dimensions stay close in 2D. Best for visualizing whether classes form clusters.

### 2.1 PCA (Principal Component Analysis)

PCA finds the **eigenvectors** of the covariance matrix — the axes that capture the most variance possible.

**How to interpret:**

*Left plot — Explained Variance:*
- Each blue bar = how much variance that component captures
- Red line = cumulative variance
- The dashed gray line marks **95%** — how many components you need to represent almost all the information
- If 2-3 components suffice → the data is low-dimensional. If you need 8+ → each feature contributes different information

*Right plot — Biplot:*
- The **points** are data samples projected onto the first two PCs, colored by class
- The **red arrows** are the original features projected onto the same space
- Long arrow = the feature has strong influence on those PCs
- Arrows pointing in the same direction = correlated features
- Arrows pointing in opposite directions = anti-correlated features
- If arrows of "commercial" features (shop_density, amenity_density) point toward the Commercial cluster → the model is learning sensible patterns

In [ ]:
pca = PCA()
X_pca = pca.fit_transform(X_scaled)
exp_var = pca.explained_variance_ratio_

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Explained variance
axes[0].bar(range(1, len(exp_var)+1), exp_var, alpha=0.6, label="Individual")
axes[0].plot(range(1, len(exp_var)+1), np.cumsum(exp_var), "ro-", label="Cumulative")
axes[0].set_xlabel("Component"); axes[0].set_ylabel("Explained Variance Ratio")
axes[0].set_title("PCA Explained Variance"); axes[0].legend()
axes[0].axhline(y=0.95, color="gray", linestyle="--", alpha=0.5)

# Biplot (PC1 vs PC2)
for i, cls in enumerate(class_names):
    mask = y_encoded == i
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], alpha=0.3, s=10, label=cls)

loadings = pca.components_[:2].T
scale = 3
for j, feat in enumerate(available_features):
    axes[1].annotate("", xy=(loadings[j,0]*scale, loadings[j,1]*scale), xytext=(0,0),
                     arrowprops=dict(arrowstyle="->", color="red", lw=1.5))
    axes[1].text(loadings[j,0]*scale*1.1, loadings[j,1]*scale*1.1, feat, fontsize=7, color="red")
axes[1].set_xlabel(f"PC1 ({exp_var[0]:.1%})"); axes[1].set_ylabel(f"PC2 ({exp_var[1]:.1%})")
axes[1].set_title("PCA Biplot"); axes[1].legend(markerscale=3)
plt.tight_layout()
plt.savefig("outputs/04_pca.png", bbox_inches="tight")
plt.show()

# How many components for 95%?
n_95 = np.argmax(np.cumsum(exp_var) >= 0.95) + 1
print(f"Components for 95% variance: {n_95} of {len(exp_var)}")
print(f"Top 3 components explain: {np.sum(exp_var[:3]):.1%}")

### 2.2 ICA (Independent Component Analysis)

ICA finds statistically **independent** signals hidden in your features. While PCA looks for directions of maximum variance (linear correlation), ICA looks for truly independent sources (statistical independence).

**Analogy:** Imagine a party with 5 simultaneous conversations and 10 microphones. PCA would find the noisiest directions. ICA would find the 5 individual conversations.

**How to interpret:**

*Left/center plots — IC1 vs IC2:*
- If classes separate along some IC axis → that independent component captures class-relevant information
- If classes overlap completely → the independent sources are not aligned with Commercial/Residential

*Right plot — Mixing Matrix:*
- Shows how each original feature contributes to each IC
- Dark red = strong positive contribution. Dark blue = strong negative contribution.
- If one IC has strong weights on shop_density and amenity_density → that IC captures "commercial activity" as a hidden signal

In [ ]:
n_components_ica = min(5, len(available_features))
ica = FastICA(n_components=n_components_ica, random_state=42, max_iter=1000)
X_ica = ica.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# IC1 vs IC2 colored by zone type
for i, cls in enumerate(class_names):
    mask = y_encoded == i
    axes[0].scatter(X_ica[mask, 0], X_ica[mask, 1], alpha=0.3, s=10, label=cls)
axes[0].set_xlabel("IC1"); axes[0].set_ylabel("IC2")
axes[0].set_title("ICA — by Zone Type"); axes[0].legend(markerscale=3)

# IC1 vs IC2 colored by city
city_list_ica = sorted(df["city"].unique())
for city_name in city_list_ica:
    mask = cities == city_name
    axes[1].scatter(X_ica[mask, 0], X_ica[mask, 1], alpha=0.3, s=10, label=city_name)
axes[1].set_xlabel("IC1"); axes[1].set_ylabel("IC2")
axes[1].set_title("ICA — by City"); axes[1].legend(markerscale=3, fontsize=8)

# Mixing matrix heatmap — shows how each original feature contributes to each IC
mixing = ica.mixing_
axes[2].imshow(np.abs(mixing), cmap="YlOrRd", aspect="auto")
axes[2].set_xticks(range(n_components_ica))
axes[2].set_xticklabels([f"IC{i+1}" for i in range(n_components_ica)])
axes[2].set_yticks(range(len(available_features)))
axes[2].set_yticklabels(available_features, fontsize=8)
axes[2].set_title("ICA Mixing Matrix (feature contributions)")
for i in range(len(available_features)):
    for j in range(n_components_ica):
        axes[2].text(j, i, f"{mixing[i,j]:.2f}", ha="center", va="center", fontsize=7)

plt.tight_layout()
plt.savefig("outputs/05_ica.png", bbox_inches="tight")
plt.show()

# Identify redundant features (high correlation between ICA-reconstructed signals)
print("ICA component kurtosis (higher = more non-Gaussian = more informative):")
for i in range(n_components_ica):
    k = kurtosis(X_ica[:, i])
    print(f"  IC{i+1}: kurtosis = {k:.2f}")

### Target Encoding: Label vs One-Hot

Demonstrates the conceptual difference between the two ways of encoding the Y variable (label):
- **Label Encoding** (y = 0, 1): Used by sklearn and tree-based models.
- **One-Hot Encoding** (y = [[1,0], [0,1]]): Used by neural networks with softmax in the output layer.

For tree-based models (RF, XGBoost), One-Hot of the target does not apply. It only matters in Section 4.5 (ANN).

### 2.3 t-SNE (t-distributed Stochastic Neighbor Embedding)

t-SNE is a **nonlinear** projection that preserves neighborhood relationships: points that are similar in high dimensions will be close in 2D.

**Key differences with PCA:**
- PCA is linear and deterministic. t-SNE is nonlinear and stochastic (that is why we fix `random_state=42`)
- PCA preserves global distances. t-SNE preserves local neighborhoods
- PCA can be used for model training. t-SNE **NEVER** — only for visualization
- The `perplexity` parameter controls how many neighbors to consider (5-50, we use 30)

**How to interpret:**
- **Colored by class (left):** If Commercial and Residential form separate clusters → the features capture real differences. If mixed → the classes are hard to separate (consistent with low accuracy).
- **Colored by city (right):** If each city forms its own cluster → the features vary more between cities than between classes (problematic). If cities mix together → the model is learning universal urban patterns, not city-specific artifacts.

In [ ]:
# t-SNE (subsample for large datasets — t-SNE is O(n²))
TSNE_MAX = 8000
if len(X_scaled) > TSNE_MAX:
    rng_tsne = np.random.RandomState(42)
    tsne_idx = rng_tsne.choice(len(X_scaled), TSNE_MAX, replace=False)
    X_tsne_input = X_scaled[tsne_idx]
    y_tsne = y_encoded[tsne_idx]
    cities_tsne = cities[tsne_idx]
    print(f"t-SNE: subsampled {TSNE_MAX} from {len(X_scaled)}")
else:
    X_tsne_input = X_scaled
    y_tsne = y_encoded
    cities_tsne = cities

perplexity = min(TSNE_PERPLEXITY, len(X_tsne_input) - 1)
if perplexity < TSNE_PERPLEXITY:
    print(f"WARNING: Dataset too small for perplexity={TSNE_PERPLEXITY}, using {perplexity}")

tsne = TSNE(n_components=2, perplexity=perplexity, random_state=42, max_iter=1000)
X_tsne = tsne.fit_transform(X_tsne_input)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, cls in enumerate(class_names):
    mask = y_tsne == i
    axes[0].scatter(X_tsne[mask, 0], X_tsne[mask, 1], alpha=0.4, s=10, label=cls)
axes[0].set_title("t-SNE — by Zone Type"); axes[0].legend(markerscale=3)

city_list_tsne = sorted(df["city"].unique())
for city_name in city_list_tsne:
    mask = cities_tsne == city_name
    axes[1].scatter(X_tsne[mask, 0], X_tsne[mask, 1], alpha=0.4, s=10, label=city_name)
axes[1].set_title("t-SNE — by City"); axes[1].legend(markerscale=3)

plt.tight_layout()
plt.savefig("outputs/06_tsne.png", bbox_inches="tight")
plt.show()

## 3. Encoding Experiments

**Encoding** is how you represent data numerically before feeding it to the model. Two key decisions:

**1. Feature scaling (X):**
- **StandardScaler**: Subtracts mean and divides by standard deviation → distribution with mean=0, std=1. Recommended for PCA and linear models.
- **MinMaxScaler**: Compresses everything between 0 and 1. Useful for neural networks and SOMs.
- **No scaling**: Raw data. Some models (trees) do not need it, but others (SVM, LR) do.
- **Log + StandardScaler**: Applies logarithm first (useful if data has log-normal distributions like density values).

**2. Target encoding (Y):**
- **Label Encoding**: y = [0, 1, 2...]. Standard for sklearn classifiers.
- **One-Hot Encoding**: y = [[1,0], [0,1]]. Required for neural network softmax output.

This section tests all scaling strategies with Logistic Regression (the most sensitive to scaling) and reports which one gives the best accuracy.

In [ ]:
# Experiment: StandardScaler vs MinMaxScaler vs No Scaling
# Using Logistic Regression as baseline
encoding_results = {}

# 1. StandardScaler (mean=0, std=1)
pipe_std = Pipeline([("scaler", StandardScaler()),
                     ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))])
scores_std = cross_val_score(pipe_std, X, y_encoded, cv=StratifiedKFold(N_FOLDS), scoring="accuracy")
encoding_results["StandardScaler"] = scores_std.mean()

# 2. MinMaxScaler (0 to 1)
pipe_mm = Pipeline([("scaler", MinMaxScaler()),
                    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))])
scores_mm = cross_val_score(pipe_mm, X, y_encoded, cv=StratifiedKFold(N_FOLDS), scoring="accuracy")
encoding_results["MinMaxScaler"] = scores_mm.mean()

# 3. No scaling (raw features)
pipe_raw = Pipeline([("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))])
scores_raw = cross_val_score(pipe_raw, X, y_encoded, cv=StratifiedKFold(N_FOLDS), scoring="accuracy")
encoding_results["No Scaling"] = scores_raw.mean()

# 4. Log-transform + StandardScaler (for skewed features)
X_log = np.log1p(np.abs(X))  # log(1+|x|) handles zeros and negatives
pipe_log = Pipeline([("scaler", StandardScaler()),
                     ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))])
scores_log = cross_val_score(pipe_log, X_log, y_encoded, cv=StratifiedKFold(N_FOLDS), scoring="accuracy")
encoding_results["Log + StandardScaler"] = scores_log.mean()

print(f"Encoding/Scaling comparison (Logistic Regression, {N_FOLDS}-fold CV):")
for name, acc in sorted(encoding_results.items(), key=lambda x: -x[1]):
    print(f"  {name:<25s} {acc:.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
names = list(encoding_results.keys())
accs = [encoding_results[n] for n in names]
ax.bar(names, accs, color=["steelblue", "coral", "gray", "forestgreen"])
ax.set_ylim(min(accs) - 0.05, max(accs) + 0.05)
ax.set_ylabel(f"Accuracy ({N_FOLDS}-fold CV)")
ax.set_title("Effect of Feature Scaling Strategy")
for i, (n, a) in enumerate(zip(names, accs)):
    ax.text(i, a + 0.005, f"{a:.4f}", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig("outputs/07_encoding_comparison.png", bbox_inches="tight")
plt.show()

In [ ]:
# Target encoding experiment: Label vs One-Hot
# With Random Forest (which handles both)
print("Target encoding comparison:")
print("  Label Encoding: y = [0, 1] — used by tree-based models and most sklearn classifiers")
print("  One-Hot Encoding: y = [[1,0], [0,1]] — used by neural networks (softmax output)")
print()

# Label encoding + RF
rf_label = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42)
scores_label = cross_val_score(rf_label, X_scaled, y_encoded, cv=StratifiedKFold(N_FOLDS), scoring="accuracy")

# One-Hot encoding doesn't apply to sklearn classifiers (they need integer labels)
# but it matters for neural networks. Demonstrate the conceptual difference:
print(f"  RF with Label Encoding:  {scores_label.mean():.4f} +/- {scores_label.std():.4f}")
print()
print("  Note: One-Hot encoding of the target is used in Section 4.5 (ANN)")
print("  where the output layer has one neuron per class with softmax activation.")
print("  Tree-based models (RF, XGBoost) always use integer label encoding internally.")

## 4. Supervised Classification

Here we train models that learn to predict the class (Commercial vs Residential) from the features.

**Professor's rule:** "Always start with the simplest model (Logistic Regression as baseline). Only increase complexity if the data justifies it."

**Shallow Models (few parameters, fast, interpretable):**
- **Logistic Regression** — The mandatory baseline. Linear decision boundary.
- **Random Forest** — Ensemble of decision trees. Captures nonlinearities.
- **XGBoost** — Gradient boosting. Generally the best shallow model.
- **SVC** — Support Vector Machine. Finds the maximum-margin hyperplane.

**Deep Model:**
- **ANN** — Artificial Neural Network with hidden layers. Tests whether deep nonlinearity adds value.

### 4.1 Logistic Regression (Baseline — Shallow)

Logistic Regression is always the **first model** you should try. It uses a sigmoid function to draw a **linear** decision boundary in feature space.

**Why start here?** Because it establishes a performance floor. If LR gives 80%, you know any more complex model must beat that to justify its complexity.

**How to interpret the Confusion Matrix:**
- **Diagonal** (top-left, bottom-right): correct predictions. The bluer, the better.
- **Off-diagonal**: errors.
  - Top-right: Commercial predicted as Residential (missed commercial activity)
  - Bottom-left: Residential predicted as Commercial (false alarm)
- If one row is mostly off-diagonal → the model struggles with that class

In [ ]:
lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

print("Logistic Regression Results:")
print(classification_report(y_test, y_pred_lr, target_names=class_names))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_lr, display_labels=class_names, ax=ax, cmap="Blues")
ax.set_title("Logistic Regression — Confusion Matrix")
plt.tight_layout()
plt.savefig("outputs/08_lr_confusion.png", bbox_inches="tight")
plt.show()

cv_scores_lr = cross_val_score(lr, X_scaled, y_encoded, cv=StratifiedKFold(N_FOLDS), scoring="accuracy")
print(f"Cross-val accuracy: {cv_scores_lr.mean():.3f} +/- {cv_scores_lr.std():.3f}")

### 4.2 XGBoost (Shallow, Gradient Boosting)

XGBoost builds decision trees **sequentially**: each new tree tries to correct the errors of the previous one. It is generally the best shallow model for tabular data.

**How to interpret the Feature Importance:**
- The longest bar = the feature XGBoost uses most to make decisions
- High importance features → these truly discriminate between classes
- Low importance features → may be noise (candidates for the ablation study)
- Compare with Random Forest importance — if both agree on the top features, the signal is robust

In [ ]:
if HAS_XGB:
    xgb = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                        random_state=42, eval_metric="mlogloss")
    xgb.fit(X_train, y_train)
    y_pred_xgb = xgb.predict(X_test)

    print("XGBoost Results:")
    print(classification_report(y_test, y_pred_xgb, target_names=class_names))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred_xgb, display_labels=class_names, ax=axes[0], cmap="Blues")
    axes[0].set_title("XGBoost — Confusion Matrix")

    importances = xgb.feature_importances_
    idx = np.argsort(importances)
    axes[1].barh([available_features[i] for i in idx], importances[idx], color="steelblue")
    axes[1].set_title("XGBoost — Feature Importance")
    plt.tight_layout()
    plt.savefig("outputs/09_xgb_results.png", bbox_inches="tight")
    plt.show()
else:
    print("Skipping XGBoost (not installed)")
    y_pred_xgb = None

### 4.3 Random Forest (Shallow, Tree Ensemble)

Random Forest trains many decision trees **in parallel**, each with a random sample of data and features. The final prediction is the majority vote of all trees.

**Advantages:** Robust to overfitting, provides feature importance, does not require scaling.

**How to interpret the Feature Importance:**
- Measures how much each feature reduces impurity (Gini) averaged across all trees
- Compare the ranking with XGBoost: if both agree on the top 3 features → robust signal
- If they differ a lot → there are complex interactions that each model exploits differently

In [ ]:
rf = RandomForestClassifier(n_estimators=200, max_depth=8, class_weight="balanced", random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("Random Forest Results:")
print(classification_report(y_test, y_pred_rf, target_names=class_names))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_rf, display_labels=class_names, ax=axes[0], cmap="Blues")
axes[0].set_title("Random Forest — Confusion Matrix")

importances = rf.feature_importances_
idx = np.argsort(importances)
axes[1].barh([available_features[i] for i in idx], importances[idx], color="forestgreen")
axes[1].set_title("Random Forest — Feature Importance")
plt.tight_layout()
plt.savefig("outputs/10_rf_results.png", bbox_inches="tight")
plt.show()

### 4.4 Support Vector Classification (SVC — Shallow)

SVC finds the **hyperplane** that maximizes the distance (margin) between classes. The kernel transforms the data to a higher-dimensional space where they can be linearly separable.

**Kernels tested:**
- **linear**: Straight decision boundary. Similar to Logistic Regression.
- **rbf** (Radial Basis Function): Curved boundary. Captures moderate nonlinearities.
- **poly** (Polynomial): Polynomial boundary. Captures feature interactions.

**How to interpret:**
- If linear kernel is best → the problem is linearly separable (LR would be enough)
- If rbf or poly is best → there are nonlinear relationships the linear models miss
- Compare with LR accuracy: the improvement measures "how nonlinear" the data is

### Export Predictions

Uses the best model (Random Forest) to predict ALL cells in the dataset (not just the test set). Saves a CSV per city with predictions, which will be used by the heatmap and comparison sections.

The accuracy reported here is **re-substitution** (prediction on data the model already saw during training), so it will be higher than the test set accuracy. The true generalization accuracy is the test set accuracy reported above.

In [ ]:
# SVC training (subsample for large datasets — SVC is O(n²))
SVC_TRAIN_MAX = 10000
if len(X_train) > SVC_TRAIN_MAX:
    rng_svc = np.random.RandomState(42)
    idx_svc = rng_svc.choice(len(X_train), SVC_TRAIN_MAX, replace=False)
    X_train_svc = X_train[idx_svc]
    y_train_svc = y_train[idx_svc]
    print(f"SVC: subsampled {SVC_TRAIN_MAX} from {len(X_train)} training samples")
else:
    X_train_svc = X_train
    y_train_svc = y_train

svc_results = {}
for kernel in ["rbf", "linear", "poly"]:
    svc = SVC(kernel=kernel, class_weight="balanced", random_state=42)
    svc.fit(X_train_svc, y_train_svc)
    y_pred_svc = svc.predict(X_test)
    acc = accuracy_score(y_test, y_pred_svc)
    svc_results[kernel] = {"accuracy": acc, "y_pred": y_pred_svc}
    print(f"SVC ({kernel}): accuracy = {acc:.3f}")

best_kernel = max(svc_results, key=lambda k: svc_results[k]["accuracy"])
print("Best kernel:", best_kernel)
print(classification_report(y_test, svc_results[best_kernel]["y_pred"],
                            target_names=class_names, zero_division=0))

# Confusion matrix
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, kernel in enumerate(["rbf", "linear", "poly"]):
    ConfusionMatrixDisplay.from_predictions(
        le.inverse_transform(y_test),
        le.inverse_transform(svc_results[kernel]["y_pred"]),
        display_labels=class_names, ax=axes[i], cmap="Blues"
    )
    axes[i].set_title(f"SVC ({kernel}) — acc={svc_results[kernel]['accuracy']:.3f}")
plt.suptitle("Support Vector Classification", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/11_svc_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: outputs/11_svc_results.png")

### 4.5 Artificial Neural Network — Deep Model (Keras/TensorFlow)

A neural network with multiple hidden layers. Architecture:
- **Input layer**: neurons matching the number of features
- **Hidden layers**: 64 → 32 neurons with ReLU activation + Dropout (20%)
- **Output layer**: softmax (one neuron per class, probabilities summing to 1)

**Professor's cheat sheet:**

| Problem | Output Activation | Loss |
|----------|------------------|------|
| Binary classification | sigmoid | binary_crossentropy |
| Multiclass | softmax | categorical_crossentropy |
| Regression | linear/none | MSE |

**How to interpret:**
- **Training/Validation curves**: If training accuracy keeps rising but validation flattens → overfitting. The gap between them = generalization error.
- **Confusion Matrix**: Same interpretation as LR. Compare with shallow models — if ANN is only marginally better → the extra complexity is not worth it.

In [ ]:
if HAS_TF:
    ANN_MAX = 10000
    n_classes = len(class_names)

    if len(X_train) > ANN_MAX:
        rng_ann = np.random.RandomState(42)
        ann_idx = rng_ann.choice(len(X_train), ANN_MAX, replace=False)
        X_train_ann = X_train[ann_idx]
        y_train_ann = y_train[ann_idx]
        print(f"ANN: subsampled {ANN_MAX} from {len(X_train)} training samples")
    else:
        X_train_ann = X_train
        y_train_ann = y_train

    y_train_cat = keras.utils.to_categorical(y_train_ann, n_classes)
    y_test_cat = keras.utils.to_categorical(y_test, n_classes)

    model = keras.Sequential()
    model.add(keras.layers.Input(shape=(X_train_ann.shape[1],)))
    for units in ANN_CONFIG["hidden_layers"]:
        model.add(keras.layers.Dense(units, activation="relu"))
        model.add(keras.layers.Dropout(0.2))
    model.add(keras.layers.Dense(n_classes, activation="softmax"))

    model.compile(optimizer=keras.optimizers.Adam(learning_rate=ANN_CONFIG["learning_rate"]),
                  loss="categorical_crossentropy", metrics=["accuracy"])

    print("ANN Architecture:")
    model.summary()

    ann_epochs = min(ANN_CONFIG["epochs"], 15)
    history = model.fit(X_train_ann, y_train_cat, epochs=ann_epochs,
                        validation_split=ANN_CONFIG["validation_split"],
                        batch_size=64, verbose=1)

    y_pred_ann = model.predict(X_test).argmax(axis=1)
    acc_ann = accuracy_score(y_test, y_pred_ann)
    print("ANN accuracy:", f"{acc_ann:.3f}")
    print(classification_report(y_test, y_pred_ann, target_names=class_names))

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].plot(history.history["loss"], label="train")
    axes[0].plot(history.history["val_loss"], label="val")
    axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
    axes[1].plot(history.history["accuracy"], label="train")
    axes[1].plot(history.history["val_accuracy"], label="val")
    axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred_ann, display_labels=class_names, ax=axes[2], cmap="Blues")
    axes[2].set_title("ANN acc=" + f"{acc_ann:.3f}")
    plt.tight_layout()
    plt.savefig("outputs/12_ann_results.png", bbox_inches="tight")
    plt.show()
else:
    print("Skipping ANN (tensorflow not installed)")
    acc_ann = None


### 4.6 Model Comparison

Summary table with the accuracy of each model on the test set.

**How to interpret:**
- The model with the highest accuracy is the "winner", but the difference matters:
  - <2% difference → they are equivalent, choose the simplest/most interpretable one
  - >5% difference → the more complex model is justified
- Compare LR (linear baseline) vs the rest: **the accuracy jump** measures how much nonlinearity exists in your data
- If all models give ~same result → the ceiling is in the data, not the algorithm

**Remember:** Accuracy > 80% is the professor's minimum target.

In [ ]:
results = {
    "Logistic Regression": accuracy_score(y_test, y_pred_lr),
    "Random Forest": accuracy_score(y_test, y_pred_rf),
    f"SVC ({best_kernel})": svc_results[best_kernel]["accuracy"],
}
if HAS_XGB and y_pred_xgb is not None:
    results["XGBoost"] = accuracy_score(y_test, y_pred_xgb)
if HAS_TF and acc_ann is not None:
    results["ANN (Keras)"] = acc_ann

df_results = pd.DataFrame(list(results.items()), columns=["Model", "Accuracy"])
df_results = df_results.sort_values("Accuracy", ascending=False)
print(df_results.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(df_results["Model"], df_results["Accuracy"], color="steelblue")
ax.set_xlim(0, 1)
ax.set_xlabel("Accuracy")
ax.set_title("Model Comparison")
for i, (_, row) in enumerate(df_results.iterrows()):
    ax.text(row["Accuracy"] + 0.01, i, f"{row['Accuracy']:.3f}", va="center")
plt.tight_layout()
plt.savefig("outputs/13_model_comparison.png", bbox_inches="tight")
plt.show()

### SVC Tuning + Updated Comparison

Same GridSearchCV process but for SVC (kernel, C, gamma). At the end, shows an updated comparison table that includes both default and tuned models.

**How to interpret the final table:** If "RF (tuned)" or "SVC (tuned)" clearly surpass their default versions, the tuning was worthwhile. If not, the defaults were already good.

In [ ]:
# Export predictions using best supervised model (RF by default)
best_model = rf
y_pred_all = best_model.predict(X_scaled)
df["predicted"] = le.inverse_transform(y_pred_all)
df["correct"] = df["predicted"] == df["label"]

for city_key in df["city"].unique():
    city_dir = f"csv/{city_key}"
    os.makedirs(city_dir, exist_ok=True)
    df_city = df[df["city"] == city_key].copy()
    pred_path = f"{city_dir}/07_predictions.csv"
    df_city.to_csv(pred_path, index=False, encoding="utf-8")
    acc = df_city["correct"].mean()
    print(f"{city_key}: {acc:.3f} accuracy ({len(df_city)} cells) -> {pred_path}")

## 5. Ablation Study & Hyperparameter Tuning

**Professor's rule:** "Change ONE variable at a time. Never change several things and say 'it improved'."

Two different experiments:

**Ablation Study (Section 5.1):** Remove ONE feature at a time and measure how much accuracy changes.
- If accuracy **drops** a lot without the feature → it was important
- If accuracy **rises** or does not change → it was noise and you can remove it
- This answers the professor's question: "How do you know the features you chose are the right ones?"

**Hyperparameter Tuning (Section 5.2):** Try different model configurations and find the optimal combination using GridSearchCV.

### 5.1 Feature Ablation Study

Process: for each feature, we remove it from the dataset and retrain the model (Random Forest with cross-validation). We measure the accuracy without that feature and calculate the "drop" compared to the baseline (all features).

**How to interpret the chart:**
- **Red bars (positive):** Removing that feature LOWERS accuracy → the feature is important. The longer the bar, the more important.
- **Green bars (negative):** Removing that feature RAISES accuracy → the feature was noise or confused the model. Candidate for removal.
- The feature with the longest red bar = the most important feature in the model

In [ ]:
# Ablation: remove one feature at a time, measure accuracy change
print("Feature Ablation Study (Random Forest)")
print("=" * 60)

# Baseline accuracy with all features
baseline_acc = cross_val_score(
    RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42),
    X_scaled, y_encoded, cv=StratifiedKFold(N_FOLDS), scoring="accuracy"
).mean()
print(f"Baseline (all {len(available_features)} features): {baseline_acc:.4f}\n")

ablation_results = {}
for feat_idx, feat_name in enumerate(available_features):
    # Remove this feature
    X_ablated = np.delete(X_scaled, feat_idx, axis=1)
    acc = cross_val_score(
        RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42),
        X_ablated, y_encoded, cv=StratifiedKFold(N_FOLDS), scoring="accuracy"
    ).mean()
    drop = baseline_acc - acc
    ablation_results[feat_name] = {"accuracy": acc, "drop": drop}
    direction = "DROP" if drop > 0 else "GAIN"
    print(f"  Without {feat_name:<25s} acc={acc:.4f}  ({direction}: {abs(drop):.4f})")

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
features_sorted = sorted(ablation_results, key=lambda f: ablation_results[f]["drop"], reverse=True)
drops = [ablation_results[f]["drop"] for f in features_sorted]
colors = ["red" if d > 0 else "green" for d in drops]
ax.barh(features_sorted, drops, color=colors, alpha=0.7)
ax.axvline(x=0, color="black", linewidth=0.5)
ax.set_xlabel("Accuracy Drop (positive = feature is important)")
ax.set_title("Feature Ablation — Impact on Accuracy")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("outputs/14_ablation_study.png", bbox_inches="tight")
plt.show()

print(f"\nMost important: {features_sorted[0]} (drop = {ablation_results[features_sorted[0]]['drop']:.4f})")
print(f"Least important: {features_sorted[-1]} (drop = {ablation_results[features_sorted[-1]]['drop']:.4f})")

### Interactive Maps (Folium)

Generates an interactive HTML map per city using Folium (based on Leaflet.js). You can click on each cell to see its ID, prediction, and actual label.

Maps are saved to `outputs/{city}/heatmap.html` — open them in the browser to explore.

### 5.2 Hyperparameter Tuning (GridSearchCV)

`GridSearchCV` tests **all combinations** of hyperparameters you give it and reports which one works best.

**For Random Forest, we test:**
- `n_estimators`: [50, 100, 200] — number of trees
- `max_depth`: [4, 8, 12, None] — maximum depth of each tree
- `min_samples_leaf`: [1, 3, 5] — minimum samples per leaf

**For SVC, we test:**
- `kernel`: [rbf, linear, poly] — type of decision boundary
- `C`: [0.1, 1, 10] — regularization strength (high C = less regularization)
- `gamma`: [scale, auto] — influence range of each support vector

**How to interpret:**
- `best_params_`: the winning combination
- `best_score_`: cross-validation accuracy of the winner
- If improvement over defaults is < 1% → do not bother tuning, defaults are fine

In [ ]:
# GridSearch for Random Forest
print("Hyperparameter Tuning — Random Forest (GridSearchCV)")
print("=" * 60)

param_grid_rf = {
    "n_estimators": [50, 100, 200],
    "max_depth": [4, 8, 12, None],
    "min_samples_leaf": [1, 3, 5],
}

cv_folds_tuning = min(3, N_FOLDS)
grid_rf = GridSearchCV(
    RandomForestClassifier(class_weight="balanced", random_state=42),
    param_grid_rf, cv=StratifiedKFold(cv_folds_tuning), scoring="accuracy",
    n_jobs=-1, verbose=0
)
grid_rf.fit(X_scaled, y_encoded)

print(f"Best params: {grid_rf.best_params_}")
print(f"Best CV accuracy: {grid_rf.best_score_:.4f}")
print(f"Default RF accuracy: {baseline_acc:.4f}")
print(f"Improvement: {grid_rf.best_score_ - baseline_acc:+.4f}")

# Visualize top 10 parameter combinations
results_df = pd.DataFrame(grid_rf.cv_results_)
results_df = results_df.sort_values("rank_test_score").head(10)
fig, ax = plt.subplots(figsize=(10, 5))
labels = [str(p) for p in results_df["params"]]
labels = [l.replace("'", "").replace("{", "").replace("}", "") for l in labels]
ax.barh(range(len(labels)), results_df["mean_test_score"], xerr=results_df["std_test_score"],
        color="steelblue", alpha=0.7)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=7)
ax.set_xlabel("Mean CV Accuracy")
ax.set_title("Top 10 Hyperparameter Combinations (RF)")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("outputs/15_tuning_rf.png", bbox_inches="tight")
plt.show()

### Feature Means Comparison Across Cities

Shows the average values of each feature per city in two formats:

**Left plot — Z-score normalized:** Each feature is normalized (global mean = 0, std = 1) so all are comparable on the same scale. Positive values = that city is above the global average for that feature. Negative values = below.

**Right plot — Raw scale:** Original values without normalization. Useful for seeing absolute magnitudes, but features like `total_bldg_area` (~900K) dominate the scale and make others invisible.

**How to interpret:** Cities with similar profiles (same pattern of high/low bars) should behave similarly in the model. If one city has a radically different profile, the model may struggle with it.

In [ ]:
# GridSearch for SVC (subsampled — SVC is O(n²), too slow on full dataset)
print("Hyperparameter Tuning — SVC (GridSearchCV, subsampled)")
print("=" * 60)

SVC_SUBSAMPLE = 5000
if len(X_scaled) > SVC_SUBSAMPLE:
    rng = np.random.RandomState(42)
    idx_sub = rng.choice(len(X_scaled), SVC_SUBSAMPLE, replace=False)
    X_svc_tune = X_scaled[idx_sub]
    y_svc_tune = y_encoded[idx_sub]
    print(f"Subsampled {SVC_SUBSAMPLE} from {len(X_scaled)} for SVC tuning")
else:
    X_svc_tune = X_scaled
    y_svc_tune = y_encoded

param_grid_svc = {
    "kernel": ["rbf", "linear", "poly"],
    "C": [0.1, 1, 10],
    "gamma": ["scale", "auto"],
}

grid_svc = GridSearchCV(
    SVC(class_weight="balanced", random_state=42),
    param_grid_svc, cv=StratifiedKFold(cv_folds_tuning), scoring="accuracy",
    n_jobs=-1, verbose=0
)
grid_svc.fit(X_svc_tune, y_svc_tune)

print(f"Best params: {grid_svc.best_params_}")
print(f"Best CV accuracy: {grid_svc.best_score_:.4f}")

# Update model comparison with tuned models
print("--- Updated Model Comparison (with tuning) ---")
tuned_results = dict(results)  # copy original results
tuned_results["RF (tuned)"] = grid_rf.best_score_
tuned_results["SVC (tuned)"] = grid_svc.best_score_

df_tuned = pd.DataFrame(list(tuned_results.items()), columns=["Model", "Accuracy"])
df_tuned = df_tuned.sort_values("Accuracy", ascending=False)
print(df_tuned.to_string(index=False))

## 4b. Comparison: Binary vs 3-Class Classification

Runs the classification with two category strategies for comparison:
1. **Binary (no Mixed-Use):** Only Commercial and Residential. Mixed-Use cells are excluded from training.
2. **3-Class:** Commercial, Mixed-Use, and Residential as separate classes.

**Why compare?** K-Means found 3 natural clusters (k=3), suggesting Mixed-Use is a real category. But Mixed-Use is inherently fuzzy — the 40% threshold to define it is arbitrary. If binary gives >3% more accuracy, the decision boundaries are cleaner without it.

**How to interpret:**
- If binary accuracy >> 3-class → Mixed-Use contaminates the decision boundaries. Use binary for prediction.
- If they are similar → Mixed-Use is learnable. Use 3-class for richer urban analysis.

In [ ]:
# Binary vs 3-Class Classification Comparison
from config import CLASS_MAP_BINARY, CLASS_MAP_3CLASS

comparison_results = {}

for mode_name, class_map_mode in [("Binary (no Mixed-Use)", CLASS_MAP_BINARY), 
                                   ("3-Class (with Mixed-Use)", CLASS_MAP_3CLASS)]:
    # Apply class mapping
    df_mode = df_raw[df_raw["zone_type"].isin(class_map_mode)].copy()
    df_mode["label"] = df_mode["zone_type"].map(class_map_mode)
    
    if len(df_mode) < 20:
        print(f"  {mode_name}: Not enough data ({len(df_mode)} rows) — skipping")
        continue
    
    X_mode = df_mode[available_features].fillna(0).values
    y_mode = LabelEncoder().fit_transform(df_mode["label"].values)
    class_names_mode = sorted(df_mode["label"].unique())
    
    min_class = pd.Series(y_mode).value_counts().min()
    n_folds_mode = min(5, min_class)
    
    if min_class < 2:
        print(f"  {mode_name}: Smallest class has {min_class} samples — skipping")
        continue
    
    scaler_mode = StandardScaler()
    X_mode_scaled = scaler_mode.fit_transform(X_mode)
    
    # Train Random Forest with cross-validation
    rf_mode = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
    scores = cross_val_score(rf_mode, X_mode_scaled, y_mode, 
                             cv=StratifiedKFold(n_folds_mode), scoring="accuracy")
    
    comparison_results[mode_name] = {
        "accuracy": scores.mean(),
        "std": scores.std(),
        "n_samples": len(df_mode),
        "n_classes": len(class_names_mode),
        "class_dist": df_mode["label"].value_counts().to_dict(),
    }
    
    print(f"  {mode_name}: accuracy = {scores.mean():.4f} +/- {scores.std():.4f} "
          f"({len(df_mode)} samples, {len(class_names_mode)} classes)")
    print(f"    Class distribution: {df_mode['label'].value_counts().to_dict()}")

# Plot comparison
if len(comparison_results) >= 2:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    modes = list(comparison_results.keys())
    accs = [comparison_results[m]["accuracy"] for m in modes]
    stds = [comparison_results[m]["std"] for m in modes]
    
    axes[0].bar(modes, accs, yerr=stds, color=["steelblue", "coral"], capsize=5)
    axes[0].set_ylabel("Accuracy (CV)")
    axes[0].set_title("Binary vs 3-Class Accuracy")
    axes[0].set_ylim(0, 1)
    for i, (m, a) in enumerate(zip(modes, accs)):
        axes[0].text(i, a + stds[i] + 0.02, f"{a:.3f}", ha="center")
    
    # Class distribution comparison
    for i, mode in enumerate(modes):
        dist = comparison_results[mode]["class_dist"]
        axes[1].bar([f"{mode}\n{k}" for k in dist.keys()], dist.values(),
                    color=["steelblue", "coral", "green"][:len(dist)])
    axes[1].set_title("Class Distribution per Mode")
    axes[1].set_ylabel("Count")
    
    plt.tight_layout()
    plt.savefig("outputs/20_binary_vs_3class.png", bbox_inches="tight")
    plt.show()
    
    # Determine winner
    best_mode = max(comparison_results, key=lambda m: comparison_results[m]["accuracy"])
    delta = abs(comparison_results[modes[0]]["accuracy"] - comparison_results[modes[1]]["accuracy"])
    print(f"\nBest mode: {best_mode}")
    print(f"Accuracy difference: {delta:.4f}")
    if delta < 0.02:
        print("Difference < 2% — modes are practically equivalent")
    elif delta < 0.05:
        print("Difference 2-5% — moderate advantage for the winner")
    else:
        print("Difference > 5% — significant advantage for the winner")

## 6. K-Means Clustering (Unsupervised)

K-Means groups the data into K clusters WITHOUT using labels. This verifies whether the natural structure of the data matches our classes.

**Professor's purpose:** "If K-Means separates the classes without supervision → the features are clear."

**How to interpret:**

*Elbow Method (left):*
- Inertia = sum of distances to centroid. Always decreases as K increases.
- The "elbow" is where the curve stops dropping quickly → the optimal K.
- If there is no clear elbow → the clusters are not well-defined.

*Silhouette Score (right):*
- Measures how well-separated the clusters are. Range: -1 to 1.
- Values > 0.5 = good separation. 0.25-0.5 = moderate. < 0.25 = overlapping clusters.
- The K with the highest silhouette = the "natural" number of groups in the data.

In [ ]:
inertias = []
sil_scores = []
K_range = range(2, KMEANS_MAX_K + 1)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_km = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels_km))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(list(K_range), inertias, "bo-")
axes[0].set_xlabel("k"); axes[0].set_ylabel("Inertia")
axes[0].set_title("Elbow Method")

axes[1].plot(list(K_range), sil_scores, "ro-")
axes[1].set_xlabel("k"); axes[1].set_ylabel("Silhouette Score")
axes[1].set_title("Silhouette Score")
plt.tight_layout()
plt.savefig("outputs/16_kmeans_elbow.png", bbox_inches="tight")
plt.show()

best_k = list(K_range)[np.argmax(sil_scores)]
print(f"Best k by silhouette: {best_k}")

km_best = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = km_best.fit_predict(X_scaled)
ari = adjusted_rand_score(y_encoded, cluster_labels)
print(f"Adjusted Rand Index (k={best_k} vs actual zones): {ari:.3f}")

ct = pd.crosstab(pd.Series(cluster_labels, name="Cluster"),
                 pd.Series(le.inverse_transform(y_encoded), name="Zone"))
print("\nCluster vs Zone Type:")
print(ct)

## 7. Heatmap — Geographic Visualization

Represents the model predictions **spatially** on the map of each city. Each point is a grid cell (150m x 150m) colored by the predicted class.

**Project step 9:** "Represent evaluation graphically and geometrically — heatmaps showing predictions spatially."

**How to interpret:**
- **Red = Commercial**, **Blue = Residential**
- Predicted commercial zones should match reality (business districts, commercial centers)
- Transitions between zones should be gradual (not random jumps) — indicating the model learned spatial patterns
- Look for "islands" of one class inside the other → potential misclassifications or zones in transition

In [ ]:
HAS_COORDS = "cell_lat" in df.columns and "cell_lon" in df.columns

if HAS_COORDS:
    city_list = sorted(df["city"].unique())
    n_cities = len(city_list)

    fig, axes = plt.subplots(1, n_cities, figsize=(6 * n_cities, 6))
    if n_cities == 1:
        axes = [axes]

    zone_colors = {"Commercial": "red", "Residential": "blue", "Other": "green"}

    for i, city_key in enumerate(city_list):
        df_city = df[df["city"] == city_key]
        ax = axes[i]
        for label in df_city["predicted"].unique():
            mask = df_city["predicted"] == label
            color = zone_colors.get(label, "gray")
            ax.scatter(df_city.loc[mask, "cell_lon"], df_city.loc[mask, "cell_lat"],
                       c=color, s=3, alpha=0.5, label=label)
        ax.set_title(f"{city_key} ({len(df_city)} cells)")
        ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
        ax.legend(markerscale=4, fontsize=8)
        ax.set_aspect("equal")

    plt.suptitle("Predicted Zone Types per City", fontsize=14)
    plt.tight_layout()
    plt.savefig("outputs/17_heatmap_all_cities.png", bbox_inches="tight")
    plt.show()
else:
    city_list = sorted(df["city"].unique())
    print("Skipping spatial heatmap — columns 'cell_lat'/'cell_lon' not found in dataset")
    print(f"Available columns: {list(df.columns)}")

In [ ]:
# Interactive folium maps (one per city)
if HAS_FOLIUM and HAS_COORDS:
    for city_key in city_list:
        df_city = df[df["city"] == city_key]
        center_lat = df_city["cell_lat"].mean()
        center_lon = df_city["cell_lon"].mean()
        m = folium.Map(location=[center_lat, center_lon], zoom_start=12)

        color_map = {"Commercial": "red", "Residential": "blue", "Other": "green"}
        id_col = "cell_id" if "cell_id" in df_city.columns else None
        for _, row in df_city.iterrows():
            cell_label = f"{row[id_col]}: " if id_col else ""
            folium.CircleMarker(
                location=[row["cell_lat"], row["cell_lon"]],
                radius=3,
                color=color_map.get(row["predicted"], "gray"),
                fill=True, fill_opacity=0.6,
                popup=f"{cell_label}{row['predicted']} (actual: {row['label']})"
            ).add_to(m)

        os.makedirs(f"outputs/{city_key}", exist_ok=True)
        map_path = f"outputs/{city_key}/heatmap.html"
        m.save(map_path)
        print(f"Saved interactive map: {map_path}")
elif not HAS_COORDS:
    print("Skipping interactive maps — no coordinate columns")
else:
    print("Skipping interactive maps (folium not installed)")

## 8. Cross-City Comparison

Compares model performance and dataset characteristics across all included cities.

**Three charts:**
1. **Accuracy per City:** In which city does the model perform best/worst? Cities with cleaner data or clearer patterns will have higher accuracy.
2. **Cell Count per City:** How many cells does each city contribute? Cities with more cells dominate the training.
3. **Class Balance per City:** Is the Commercial/Residential imbalance consistent across cities? If one city has 50/50 and another 90/10, the model may be biased toward the more common city's patterns.

---

**Pipeline complete.** All plots saved to `outputs/`.

**Summary of generated files:**
- `outputs/01-03` — EDA (distribution, boxplots, correlation)
- `outputs/03b-03c` — SOM (U-Matrix, component planes)
- `outputs/04-06` — Dimensionality reduction (PCA, ICA, t-SNE)
- `outputs/07` — Encoding/scaling experiment
- `outputs/08-12` — Supervised models (LR, XGB, RF, SVC, ANN)
- `outputs/13` — Model comparison
- `outputs/14-15` — Ablation study + hyperparameter tuning
- `outputs/16` — K-Means clustering
- `outputs/17` — Geographic heatmap
- `outputs/18-19` — Cross-city comparison and feature means
- `outputs/20` — Binary vs 3-Class comparison
- `outputs/21` — Transfer Learning (Ground Truth vs OSM-Only)

In [ ]:
feat_means = df.groupby("city")[available_features].mean()
feat_means_norm = (feat_means - feat_means.mean()) / (feat_means.std() + 1e-8)

fig, axes = plt.subplots(1, 2, figsize=(20, 7))

# Left: Normalized (z-score) — all features comparable
feat_means_norm.T.plot.bar(ax=axes[0])
axes[0].set_title("Feature Means per City (z-score normalized)")
axes[0].set_xlabel("Feature")
axes[0].set_ylabel("Z-Score (0 = global mean)")
axes[0].legend(title="City", fontsize=8)
axes[0].axhline(y=0, color="black", linewidth=0.5, linestyle="--")
axes[0].tick_params(axis="x", rotation=45)

# Right: Raw scale (reference)
feat_means.T.plot.bar(ax=axes[1])
axes[1].set_title("Feature Means per City (raw scale)")
axes[1].set_xlabel("Feature")
axes[1].set_ylabel("Mean Value")
axes[1].legend(title="City", fontsize=8)
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig("outputs/19_feature_means.png", bbox_inches="tight")
plt.show()

print("\nNormalized feature means per city (z-scores):")
print(feat_means_norm.round(2).to_string())

## 9. Transfer Learning: Ground Truth vs OSM-Only

**Central project question:** Can a model trained with cities that have property data (Ground Truth) predict zones in cities that only have OpenStreetMap data?

**Experimental design:**
- **Group A (Ground Truth):** NYC, Philadelphia, Chicago — zone_type comes from PLUTO/OPA/Cook County
- **Group B (OSM-Only):** DC, SF, LA — zone_type comes from OSM landuse polygons

**Process:**
1. Train model ONLY with Group A data
2. Evaluate on Group A test set → reference accuracy
3. Predict on Group B → transfer accuracy
4. Compare both → the delta measures how much is lost by relying only on OSM

**How to interpret:**
- If Group B accuracy > 80% → "OSM is sufficient to predict urban zones"
- If Group B accuracy << Group A → "Property data provides irreplaceable signal"
- Either way, it is a publishable finding that directly answers the research question

In [ ]:
# Transfer Learning: Train on Ground Truth, Predict OSM-Only
from config import CITY_REGISTRY

# Identify groups
gt_cities = [k for k, v in CITY_REGISTRY.items() if v.get("group") == "ground_truth"]
osm_cities = [k for k, v in CITY_REGISTRY.items() if v.get("group") == "osm_only"]

print(f"Ground Truth cities: {gt_cities}")
print(f"OSM-Only cities: {osm_cities}")

# Split data by group
df_gt = df[df["city"].isin(gt_cities)]
df_osm = df[df["city"].isin(osm_cities)]

print(f"\nGround Truth: {len(df_gt)} cells across {df_gt['city'].nunique()} cities")
print(f"OSM-Only: {len(df_osm)} cells across {df_osm['city'].nunique()} cities")

if len(df_gt) >= 20 and len(df_osm) >= 10:
    # Prepare Ground Truth data
    X_gt = df_gt[available_features].fillna(0).values
    y_gt = le.transform(df_gt["label"].values)
    
    scaler_gt = StandardScaler()
    X_gt_scaled = scaler_gt.fit_transform(X_gt)
    
    # Train/test split within Ground Truth
    X_gt_train, X_gt_test, y_gt_train, y_gt_test = train_test_split(
        X_gt_scaled, y_gt, test_size=0.2, random_state=42, 
        stratify=y_gt if pd.Series(y_gt).value_counts().min() >= 2 else None
    )
    
    # Train RF on Ground Truth
    rf_transfer = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42)
    rf_transfer.fit(X_gt_train, y_gt_train)
    
    # Evaluate on Ground Truth test set
    acc_gt_test = accuracy_score(y_gt_test, rf_transfer.predict(X_gt_test))
    print(f"\nGround Truth test accuracy: {acc_gt_test:.3f}")
    
    # Predict OSM-Only (zero-shot transfer)
    X_osm = df_osm[available_features].fillna(0).values
    y_osm = le.transform(df_osm["label"].values)
    X_osm_scaled = scaler_gt.transform(X_osm)  # Use GT scaler!
    
    y_pred_osm = rf_transfer.predict(X_osm_scaled)
    acc_osm = accuracy_score(y_osm, y_pred_osm)
    print(f"OSM-Only transfer accuracy: {acc_osm:.3f}")
    print(f"Delta (GT - OSM): {acc_gt_test - acc_osm:+.3f}")
    
    # Per-city breakdown
    print("\nPer-city accuracy:")
    transfer_acc = {}
    for city in sorted(df["city"].unique()):
        df_city_tr = df[df["city"] == city]
        X_city = df_city_tr[available_features].fillna(0).values
        y_city = le.transform(df_city_tr["label"].values)
        X_city_scaled = scaler_gt.transform(X_city)
        y_pred_city = rf_transfer.predict(X_city_scaled)
        acc = accuracy_score(y_city, y_pred_city)
        group = "GT" if city in gt_cities else "OSM"
        transfer_acc[city] = {"accuracy": acc, "group": group, "n": len(df_city_tr)}
        print(f"  [{group}] {city:<15s} {acc:.3f}  ({len(df_city_tr)} cells)")
    
    # Visualization
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    
    # Plot 1: Accuracy by group
    acc_by_group = {"Ground Truth": acc_gt_test, "OSM-Only": acc_osm}
    colors_group = ["steelblue", "coral"]
    axes[0].bar(acc_by_group.keys(), acc_by_group.values(), color=colors_group)
    axes[0].set_ylim(0, 1)
    axes[0].set_ylabel("Accuracy")
    axes[0].set_title("Ground Truth vs OSM-Only Accuracy")
    for i, (g, a) in enumerate(acc_by_group.items()):
        axes[0].text(i, a + 0.02, f"{a:.3f}", ha="center", fontsize=12, fontweight="bold")
    axes[0].axhline(y=0.8, color="gray", linestyle="--", alpha=0.5, label="80% threshold")
    axes[0].legend()
    
    # Plot 2: Per-city accuracy colored by group
    cities_sorted_tr = sorted(transfer_acc, key=lambda c: transfer_acc[c]["accuracy"], reverse=True)
    city_accs = [transfer_acc[c]["accuracy"] for c in cities_sorted_tr]
    city_colors = ["steelblue" if transfer_acc[c]["group"] == "GT" else "coral" for c in cities_sorted_tr]
    axes[1].bar(cities_sorted_tr, city_accs, color=city_colors)
    axes[1].set_ylim(0, 1)
    axes[1].set_ylabel("Accuracy")
    axes[1].set_title("Per-City Accuracy (trained on GT only)")
    for i, c in enumerate(cities_sorted_tr):
        axes[1].text(i, transfer_acc[c]["accuracy"] + 0.02, f"{transfer_acc[c]['accuracy']:.3f}", 
                     ha="center", fontsize=9)
    # Legend
    from matplotlib.patches import Patch
    axes[1].legend(handles=[Patch(color="steelblue", label="Ground Truth"), 
                            Patch(color="coral", label="OSM-Only")], fontsize=9)
    
    # Plot 3: Confusion matrix for OSM-Only predictions
    ConfusionMatrixDisplay.from_predictions(y_osm, y_pred_osm, display_labels=class_names, 
                                            ax=axes[2], cmap="Blues")
    axes[2].set_title(f"OSM-Only Predictions (acc={acc_osm:.3f})")
    
    plt.tight_layout()
    plt.savefig("outputs/21_transfer_learning.png", bbox_inches="tight")
    plt.show()
    
    # Summary
    print(f"\n{'='*60}")
    if acc_osm >= 0.80:
        print("OSM-Only accuracy >= 80% — OSM data is sufficient for urban zone prediction")
    else:
        print("OSM-Only accuracy < 80% — property data provides irreplaceable signal")
    print(f"  Ground Truth: {acc_gt_test:.1%} | OSM-Only: {acc_osm:.1%} | Delta: {acc_gt_test-acc_osm:+.1%}")
else:
    if len(df_osm) < 10:
        print("Skipping transfer learning — no OSM-only cities in dataset")
        print(f"Run `python run_pipeline.py` with all 6 cities to enable this section")
    else:
        print("Skipping transfer learning — not enough Ground Truth data")

---

**Analysis complete.** All results, plots, and transfer learning metrics have been generated. See `experiment_log.md` for the full documentation of iterations, decisions, and findings.